# CSV to database preparation

This workflow prepares the book dataset for ingestion: load the source, inspect and clean it, then check that its columns and types fit the existing database table.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(exist_ok=True)

In [3]:
from sqlalchemy import create_engine

SUPABASE_DATABASE_URL = os.getenv("SUPABASE_DATABASE_URL")
engine = create_engine(SUPABASE_DATABASE_URL)


In [4]:
import pandas as pd

df = pd.read_csv(DATA_DIR / "books_with_emotions.csv")

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5197 entries, 0 to 5196
Data columns (total 21 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   isbn13              5197 non-null   int64  
 1   isbn10              5197 non-null   str    
 2   title               5197 non-null   str    
 3   authors             5165 non-null   str    
 4   categories          5167 non-null   str    
 5   thumbnail           5031 non-null   str    
 6   description         5197 non-null   str    
 7   published_year      5197 non-null   float64
 8   average_rating      5197 non-null   float64
 9   num_pages           5197 non-null   float64
 10  ratings_count       5197 non-null   float64
 11  title_and_subtitle  5197 non-null   str    
 12  tagged_description  5197 non-null   str    
 13  simple_categories   5197 non-null   str    
 14  anger               5197 non-null   float64
 15  disgust             5197 non-null   float64
 16  fear             

## Inspect the source data

Review the dataset structure, sample records, size, missing values, columns, and inferred types. These checks help spot issues before preparing records for the database.

In [6]:
df.head()

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,...,title_and_subtitle,tagged_description,simple_categories,anger,disgust,fear,joy,sadness,surprise,neutral
0,9780002005883,0002005883,Gilead,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,A NOVEL THAT READERS and critics have been eag...,2004.0,3.85,247.0,...,Gilead,9780002005883 A NOVEL THAT READERS and critics...,Fiction,0.064134,0.273591,0.928168,0.932797,0.646217,0.967158,0.729603
1,9780002261982,0002261987,Spider's Web,Charles Osborne;Agatha Christie,Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...,A new 'Christie for Christmas' -- a full-lengt...,2000.0,3.83,241.0,...,Spider's Web: A Novel,9780002261982 A new 'Christie for Christmas' -...,Fiction,0.612618,0.348285,0.942528,0.704421,0.887940,0.111690,0.252545
2,9780006178736,0006178731,Rage of angels,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,"A memorable, mesmerizing heroine Jennifer -- b...",1993.0,3.93,512.0,...,Rage of angels,"9780006178736 A memorable, mesmerizing heroine...",Fiction,0.064134,0.104007,0.972321,0.767237,0.549477,0.111690,0.078766
3,9780006280897,0006280897,The Four Loves,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=XhQ5X...,Lewis' work on the nature of love divides love...,2002.0,4.15,170.0,...,The Four Loves,9780006280897 Lewis' work on the nature of lov...,Nonfiction,0.351483,0.150723,0.360707,0.251881,0.732686,0.111690,0.078766
4,9780006280934,0006280935,The Problem of Pain,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=Kk-uV...,"""In The Problem of Pain, C.S. Lewis, one of th...",2002.0,4.09,176.0,...,The Problem of Pain,"9780006280934 ""In The Problem of Pain, C.S. Le...",Nonfiction,0.081412,0.184495,0.095043,0.040564,0.884390,0.475881,0.078766


In [7]:
df.shape

(5197, 21)

In [8]:
df.isna().sum()

isbn13                  0
isbn10                  0
title                   0
authors                32
categories             30
thumbnail             166
description             0
published_year          0
average_rating          0
num_pages               0
ratings_count           0
title_and_subtitle      0
tagged_description      0
simple_categories       0
anger                   0
disgust                 0
fear                    0
joy                     0
sadness                 0
surprise                0
neutral                 0
dtype: int64

In [9]:
df.columns

Index(['isbn13', 'isbn10', 'title', 'authors', 'categories', 'thumbnail',
       'description', 'published_year', 'average_rating', 'num_pages',
       'ratings_count', 'title_and_subtitle', 'tagged_description',
       'simple_categories', 'anger', 'disgust', 'fear', 'joy', 'sadness',
       'surprise', 'neutral'],
      dtype='str')

In [10]:
df.dtypes

isbn13                  int64
isbn10                    str
title                     str
authors                   str
categories                str
thumbnail                 str
description               str
published_year        float64
average_rating        float64
num_pages             float64
ratings_count         float64
title_and_subtitle        str
tagged_description        str
simple_categories         str
anger                 float64
disgust               float64
fear                  float64
joy                   float64
sadness               float64
surprise              float64
neutral               float64
dtype: object

## Clean and validate

Fill missing author names with a readable placeholder, then recheck missing values and ISBN-13 duplicates. This gives us a basic data quality check before loading.

In [11]:
df["authors"] = df["authors"].fillna("Unknown")
df["authors"].isna().sum()  # 0

np.int64(0)

In [12]:
df.isna().sum()

isbn13                  0
isbn10                  0
title                   0
authors                 0
categories             30
thumbnail             166
description             0
published_year          0
average_rating          0
num_pages               0
ratings_count           0
title_and_subtitle      0
tagged_description      0
simple_categories       0
anger                   0
disgust                 0
fear                    0
joy                     0
sadness                 0
surprise                0
neutral                 0
dtype: int64

In [13]:
df["isbn13"].duplicated().sum()

np.int64(0)

## Compare with the destination schema

Inspect the existing `books` table, then compare its columns and types with the DataFrame. Use the comparison to identify mismatches to resolve before inserting rows.

In [14]:
from sqlalchemy import inspect

inspector = inspect(engine)

db_columns = inspector.get_columns("books")

for col in db_columns:
    print(col["name"], col["type"], col["nullable"])

id BIGINT False
isbn13 VARCHAR True
isbn10 VARCHAR True
title VARCHAR False
authors TEXT True
categories TEXT True
thumbnail TEXT True
description TEXT True
published_year INTEGER True
average_rating DOUBLE PRECISION True
num_pages INTEGER True
ratings_count INTEGER True
title_and_subtitle TEXT True
tagged_description TEXT True
simple_categories TEXT True
anger DOUBLE PRECISION True
disgust DOUBLE PRECISION True
fear DOUBLE PRECISION True
joy DOUBLE PRECISION True
sadness DOUBLE PRECISION True
surprise DOUBLE PRECISION True
neutral DOUBLE PRECISION True


In [15]:
comparison = pd.DataFrame([
    {
        "postgres_column": col["name"],
        "pandas_column": str(col["name"])
            if col["name"] in df.columns else "Missing",
        "postgres_type": str(col["type"]),
        "pandas_type": str(df[col["name"]].dtype)
            if col["name"] in df.columns else "Missing",
        "nullable": col["nullable"],
    }
    for col in db_columns
])

comparison

,postgres_column,pandas_column,postgres_type,pandas_type,nullable
0,id,Missing,BIGINT,Missing,False
1,isbn13,isbn13,VARCHAR,int64,True
2,isbn10,isbn10,VARCHAR,str,True
3,title,title,VARCHAR,str,False
4,authors,authors,TEXT,str,True
5,categories,categories,TEXT,str,True
6,thumbnail,thumbnail,TEXT,str,True
7,description,description,TEXT,str,True
8,published_year,published_year,INTEGER,float64,True
9,average_rating,average_rating,DOUBLE PRECISION,float64,True
